# B2-023 — Practice p09

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** intro · **Time budget:** 50 minutes  
**Concepts:** denoising-diffusion-probabilistic-models

## Task

Implement the unit's linear variance schedule and the closed-form forward sampler.

**Contract.** `make_schedule(T, beta_1, beta_T)` returns a dict with keys `"beta"`, `"alpha"`, `"alpha_bar"`, `"beta_tilde"`, each a float64 tensor of shape `(T+1,)`: `beta[0] = 0` and `beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)`; `alpha = 1 - beta`; `alpha_bar = torch.cumprod(alpha, dim=0)` (so `alpha_bar[0] = 1`); `beta_tilde[0] = 0` and `beta_tilde[t] = (1 - alpha_bar[t-1]) / (1 - alpha_bar[t]) * beta[t]` for $t\ge1$ (so `beta_tilde[1] = 0`). `q_sample(x0, t, eps, alpha_bar)` returns `alpha_bar[t].to(x0.dtype).unsqueeze(1).sqrt() * x0 + (1 - alpha_bar[t].to(x0.dtype).unsqueeze(1)).sqrt() * eps` for `x0`, `eps` of shape `(B,d)` and an int64 `t` of shape `(B,)` with values in $1,\dots,T$.
`make_schedule` raises `ValueError` unless `T` is an int $\ge2$ and $0<\beta_1\le\beta_T<1$. `q_sample` returns a tensor with `x0`'s shape and dtype and raises `ValueError` if `x0` and `eps` differ in shape, if `t` is not an int64 tensor of shape `(B,)`, or if any `t` lies outside $1,\dots,T$ with `T = alpha_bar.shape[0] - 1`.

**Banned:** Python loops over rows or timesteps inside `q_sample`; drawing random numbers inside either function; `torch.distributions`.

**Probes.**

1. `make_schedule(5, 0.1, 0.5)` gives (`ATOL = RTOL = 1e-12`): `beta = [0, 0.1, 0.2, 0.3, 0.4, 0.5]`, `alpha = [1, 0.9, 0.8, 0.7, 0.6, 0.5]`, `alpha_bar = [1, 0.9, 0.72, 0.504, 0.3024, 0.1512]`, and `beta_tilde = [0, 0, 0.0714285714285714, 0.1693548387096774, 0.2844036697247707, 0.4109330819981150]`.
2. `make_schedule(50, 1e-3, 0.2)`: every entry has shape `(51,)` and dtype float64; `beta[1]` and `beta[50]` equal `1e-3` and `0.2` within `1e-15`; `alpha_bar[50]` equals `0.004505985481139705` within `1e-12`; `alpha_bar[1:]` is strictly decreasing. A local off-by-one variant whose betas are `torch.linspace(beta_1, beta_T, T + 1)[1:]` must fail the endpoint probe.
3. With the 5-step schedule and the float32 literals `X0`, `T_IDX`, `EPS` below, `q_sample(X0, T_IDX, EPS, alpha_bar)` equals `[[1.1067972183, -1.8973665237], [-0.3493078947, 1.7635102272], [-0.2045837343, 0.7980118394]]` with `atol = rtol = 1e-6` and has dtype float32. A local variant that uses `alpha` where `alpha_bar` belongs agrees on row 0 (because $t=1$) and differs by more than `0.1` on rows 1 and 2.
4. **Statistical probe.** With the 50-step schedule, `x0` equal to the float64 row `(1.0, -1.0)` repeated 200,000 times, `t = 25` for every row, and `eps = torch.randn(200_000, 2, generator=torch.Generator().manual_seed(SEED), dtype=torch.float64)`, every coordinate of the sample mean of `q_sample` is within `0.01` of `sqrt(alpha_bar[25]) * x0[0]` and every coordinate's sample variance is within `0.01` of `1 - alpha_bar[25]` (`alpha_bar[25]` is about `0.2760332057`).
5. Each invalid input named in the contract raises `ValueError`.

**Required answer-check assertions.** Probes 1–5.

In [ ]:
import torch

SEED = 20261015
X0 = torch.tensor([[1.0, -2.0], [0.5, 0.5], [-1.0, 3.0]])
T_IDX = torch.tensor([1, 3, 5])
EPS = torch.tensor([[0.5, 0.0], [-1.0, 2.0], [0.2, -0.4]])
ATOL = 1e-12
RTOL = 1e-12

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.